In [33]:
from pyfmi import load_fmu
from utils import get_git_root
root = get_git_root()

# Exploration
For now, we just want to try and identify a changing variable we might be interested in. 

In [34]:
model = load_fmu(f"{root}/Data/WindTurbine.fmu")

In [35]:
opts = model.simulate_options()
opts['solver'] = 'CVode'

In [36]:
res = model.simulate(start_time=0, final_time=3600000, options=opts)

Final Run Statistics: --- 

 Number of steps                                 : 501
 Number of function evaluations                  : 504
 Number of Jacobian evaluations                  : 9
 Number of function eval. due to Jacobian eval.  : 9
 Number of error test failures                   : 0
 Number of nonlinear iterations                  : 501
 Number of nonlinear convergence failures        : 0
 Number of state function evaluations            : 502

Solver options:

 Solver                   : CVode
 Linear multistep method  : BDF
 Nonlinear solver         : Newton
 Linear solver type       : DENSE
 Maximal order            : 5
 Tolerances (absolute)    : 1e-08
 Tolerances (relative)    : 1e-06

Simulation interval    : 0.0 - 3600000.0 seconds.
Elapsed simulation time: 0.08257710000907537 seconds.


In [37]:
var_names = model.get_model_variables().keys()
for name in var_names:
    if "P" in name:
        print(name)


lin.LossPower
sou.P.apparent
sou.P.cosPhi
tur.P
tur.load.P
weaBus.TDewPoi
lin.T_heatPort
res.P
res.P_nominal
tur.load.P_nominal
weaDat.TDewPoi
sou.P.phi
sou.P.real
tur.load.Pow
weaDat.weaBus.TDewPoi
weaDat.TDewPoiSou
lin.useHeatPort


In [38]:
var_names = model.get_model_variables().keys()
for name in var_names:
    if "P" in name:
        print(name, set(list(res[name])))

lin.LossPower {np.float64(0.0), np.float64(0.673346134766885), np.float64(0.31337701014740676), np.float64(2.6576848119649075), np.float64(1.5018642789793841), np.float64(5.94935441287864), np.float64(0.6733461347668851), np.float64(4.140337781785511), np.float64(8.084267101921812), np.float64(5.949354412878639), np.float64(7.6312352851546), np.float64(11.075688466500864), np.float64(5.561464455564696), np.float64(13.925932269793945), np.float64(13.925932269793947), np.float64(0.18328367694916475), np.float64(7.191224377321422), np.float64(17.10058363002706), np.float64(20.599179822548557), np.float64(24.421259273742404), np.float64(28.566361557007173), np.float64(33.03402738875308), np.float64(38.47634453496053), np.float64(40.531235852041256), np.float64(0.06942275192627234), np.float64(43.06712092028613), np.float64(0.0010850270610858468), np.float64(0.03905334753363529), np.float64(0.13123701934635515), np.float64(0.21255738839898478), np.float64(0.017358398992047254), np.float64(4

In [39]:
best = 0
best_name = ""
for name in var_names:
        try: diff_results = len(set(list(res[name])))
        except: print(f"{name} does not work")
        if diff_results > 1:
                print(diff_results, name)
        if diff_results > best:
                best = diff_results
                best_name = name
print(best, best_name)       

38 lin.LossPower
38 sen.I
36 sen.S[1]
49 sen.S[2]
35 sen.V
41 sou.P.apparent
40 sou.P.cosPhi
35 sou.sou.S[1]
49 sou.sou.S[2]
43 sou.sou.phi
35 sou.terminal.i[1]
50 sou.terminal.i[2]
35 tur.P
35 tur.load.P
36 tur.load.S[1]
49 tur.load.S[2]
35 tur.load.i[1]
50 tur.load.i[2]
36 tur.load.v[1]
56 tur.load.v[2]
221 weaBus.HDifHor
195 weaBus.HDirNor
221 weaBus.HGloHor
102 weaBus.HHorIR
422 weaBus.TBlaSky
66 weaBus.TDewPoi
68 weaBus.TDryBul
90 weaBus.ceiHei
501 weaBus.cloTim
12 weaBus.nOpa
12 weaBus.nTot
56 weaBus.relHum
501 weaBus.solAlt
501 weaBus.solDec
501 weaBus.solHouAng
501 weaBus.solTim
501 weaBus.solZen
38 weaBus.winDir
57 weaBus.winSpe
35 lin.terminal_n.i[1]
50 lin.terminal_n.i[2]
501 lin.terminal_n.theta[1]
35 lin.terminal_p.i[1]
50 lin.terminal_p.i[2]
501 lin.terminal_p.theta[1]
36 lin.terminal_p.v[1]
56 lin.terminal_p.v[2]
501 res.terminal.theta[1]
35 sen.terminal_n.i[1]
50 sen.terminal_n.i[2]
501 sen.terminal_n.theta[1]
36 sen.terminal_n.v[1]
56 sen.terminal_n.v[2]
35 sen.termina

So we are definitely seeing changing weather variables, but not changing the power. My guess is that the model isn't evaluating the wind speed propery

In [40]:
print(model.get_variable_causality('tur.P'))

4


In [41]:
model2 = load_fmu(f"{root}/FMU_prjs/01-Simplest_FMU/SimpleModel_fixed.fmu")

In [42]:
res = model2.simulate(start_time=0, final_time=3600000)

Simulation interval    : 0 - 3600000.0 seconds.
Elapsed simulation time: 0.04505160001281183 seconds.
